# Part 1: Implementation

### Exercise 1 - SVM for Classification

In [19]:
# You will use: Wine dataset (classification), California Housing dataset (regression)

# Exercise 1 — SVM for Classification (Wine Dataset)
# Load the Wine dataset using:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Split the dataset into training and test sets.
wine = load_wine()
wine_data = wine.data
wine_labels = wine.target
X_train, X_test, y_train, y_test = train_test_split(wine_data, wine_labels, test_size=0.2, random_state=42)



In [20]:
# Scale the features (important for SVM).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train an SVM classifier.
svm_classifier = SVC(kernel='linear')
svm_classifier.fit(X_train_scaled, y_train)

# Since SVM is fundamentally binary, use: One-vs-Rest (OvR), or One-vs-One (OvO)
svm_classifier.predict(X_test)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [21]:
# Tune key hyperparameters: C, kernel (linear, rbf), gamma (if using rbf)
# Use GridSearch
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.1, 1]
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5)
# Fit the GridSearchCV to the scaled training data
grid_search.fit(X_train_scaled, y_train)

best_estimator = grid_search.best_estimator_
best_estimator.predict(X_test)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [22]:
# Evaluate the model using: Accuracy, precision, recall
from sklearn.metrics import accuracy_score, precision_score, recall_score

X_test_scaled = scaler.transform(X_test) # Scale X_test using the fitted scaler
y_pred = best_estimator.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted') # Use weighted average for multi-class
recall = recall_score(y_test, y_pred, average='weighted') # Use weighted average for multi-class
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

# Confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix_result = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", confusion_matrix_result)

Accuracy: 0.9722222222222222
Precision: 0.9753086419753088
Recall: 0.9722222222222222
Confusion Matrix:
 [[14  0  0]
 [ 0 13  1]
 [ 0  0  8]]


In [24]:
print(grid_search.best_params_)

{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}


**Questions to Answer:
What is your best test metric to use in this case? and why?**

Accuracy provides a good overall summary of the model's correctness. Precision and Recall are also good choices to ensure that the model performs well across all classes.

**Which kernel performed best?**

linear

**How did scaling affect performance?**

Scaling is important for SVMs. SVMs calculate distances between data points, and if features have different scales, features with larger values can dominate the distance calculations.

### Exercise 2- SVM for Regression

In [8]:
# Exercise 2 — SVM for Regression (California Housing Dataset)
# Load the dataset using:
from sklearn.datasets import fetch_california_housing

# Because SVMs scale poorly with large datasets: Use a subset (~2,000 samples).
data = fetch_california_housing()
X, y = data.data, data.target

In [10]:
# Split the data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Scale the features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# Train an SVM regressor.
from sklearn.svm import SVR # Import SVR for regression
svm_regressor = SVR(kernel='linear') # Use SVR instead of SVC
svm_regressor.fit(X_train_scaled, y_train)

SVR(kernel='linear')

In [16]:
# Tune hyperparameters: C, epsilon, kernel, gamma
from sklearn.model_selection import GridSearchCV

param_grid_svr = {
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 0.5],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_search_svr = GridSearchCV(SVR(), param_grid_svr, cv=2, scoring='neg_mean_squared_error')
grid_search_svr.fit(X_train_scaled, y_train)

best_svr = grid_search_svr.best_estimator_
print("Best SVR parameters:", grid_search_svr.best_params_)
print("Best SVR estimator:", best_svr)

Best SVR parameters: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Best SVR estimator: SVR(C=10)


In [17]:
# Evaluate using: RMSE (Root Mean Squared Error) and other metrics
from sklearn.metrics import mean_squared_error
import numpy as np
RMSE = np.sqrt(mean_squared_error(y_test, best_svr.predict(scaler.transform(X_test))))
print("RMSE:", RMSE)

MSE = mean_squared_error(y_test, best_svr.predict(scaler.transform(X_test)))
print("MSE:", MSE)

RMSE: 0.5689420963805115
MSE: 0.32369510903385124


**Questions to Answer:
What is your best metric? and why?**

RMSE (Root Mean Squared Error) is generally considered a very good metric to use. It measures the average magnitude of the errors, and because it squares the errors before averaging, it penalizes larger errors more heavily. Taking the square root brings the error back to the same units as the target variable, making it more interpretable than MSE.

**Which kernel worked best?**

rbf worked best

**How sensitive was performance to C and gamma?**

The optimal C value was 10 indicates the regularization strength. It was sensitive to c, and worked better with a larger C. Reducing C could have resulted in underfitting.

The model's complexity and ability to fit the training data were quite sensitive to how spread out the influence of individual training examples was. Too high a gamma could lead to overfitting (very wiggly boundaries), while too low could lead to underfitting (too smooth boundaries).

# Part 2: Reflection

**1. What did you learn about SVM?
Consider:
Sensitivity to scaling
Effect of hyperparameters
Kernel trick behavior
Margin intuition**

SVMs work by finding the optimal hyperplane that maximizes the margin between classes. Scaling the data is important, because if some features have larger numerical ranges, they can disproportionately influence the distance calculations.

A small C value creates a larger margin but may allow more misclassifications, leading to a simpler model (potentially underfitting). A large C aims to classify all training examples correctly, resulting in a narrower margin and a more complex model (potentially overfitting).

For the kernal hyperparameter, a linear kernel finds a straight hyperplane, which works for data that is linearly separable. The rbf kernel allows for non-linear decision boundaries to be found by mapping the data into more dimensions.

The gamma value defines the influence of a single training example. A small gamma means a large influence, resulting in a smoother decision boundary (potentially underfitting). A large gamma means a small influence, leading to a more complex, wiggly decision boundary (potentially overfitting).

Epsilon defines a margin of tolerance where no penalty is associated with errors. A larger epsilon means a wider margin, allowing more errors within the margin, leading to a simpler model. A smaller epsilon means a tighter margin, penalizing even small errors, leading to a more complex model.

The kernel trick allows you to find non-linear decision boundaries  by computing the similarity (dot product) between data points in that higher-dimensional space using a kernel function.

Larger margin leads to better generalization to unseen data, making the model more robust and less prone to overfitting.

**2. When might ensemble methods be more beneficial than SVM?
Think about:
Large datasets
Non-linear boundaries
Interpretability
Computational cost
Stability**

SVMs can become computationally expensive and slow when dealing with large datasets. Usually ensemble methods can handle large datasets more efficiently, as they are well-suited for parallelization.

When non-linearity is extremely intricate or high-dimensional, ensemble methods might be able to capture the patterns more flexibly without requiring as much tuning.

It is difficult to interpret how individual features contribute to a prediction for SVMs. Many ensemble methods are also complex, but some can offer more interpretability.

SVM training can be very slow for large datasets. Ensemble methods, particularly decision tree-based ones, can often train faster and are more easily parallelized, which can be a significant advantage in terms of computational efficiency and time-to-solution.

Ensemble methods generally can have more stability and robustness. By combining predictions from multiple base models, they can be less sensitive to noise or outliers in the data compared to a single, complex SVM model.

**3. Any observations or surprises?
For example:
Did RBF outperform linear?
Did regression perform worse than expected?
Was tuning expensive?
Did SVM overfit easily?**

Not many surprises that I noticed, but RBF did outperform linear. The RMSE of approximately 0.569 for the California Housing dataset seems like a reasonable result.

Hyperparameter tuning using GridSearchCV was computationally expensive and made it take a long time to run the code.

